### Data Build to Merge HCRIS Data with KFF Data

In [2]:
import pandas as pd
import numpy as np

In [3]:
# Load KFF Data
kff_dat = pd.read_csv('../../../data/input/kff_expansion_data.csv')
print('KFF shape:', kff_dat.shape)
kff_dat.head()

KFF shape: (51, 6)


,Location,Status of Medicaid Expansion Decision,Expansion Implementation Date,Expansion Adopted Through Ballot Initiative,Trigger Law In Place,Footnotes
0,Alabama,Not Adopted,NaN,NaN,NaN,NaN
1,Alaska,Adopted,9/1/2015,No,No,NaN
2,Arizona,Adopted,1/1/2014,No,Yes,1
3,Arkansas,Adopted,1/1/2014,No,Yes,1
4,California,Adopted,1/1/2014,No,No,NaN


In [5]:
# Cleaning the Data
kff_final = kff_dat.copy()

# Create expanded indicator
kff_final['expanded'] = kff_final['Status of Medicaid Expansion Decision'] == 'Adopted'

# Parse adoption date
kff_final['date_adopted'] = pd.to_datetime(
    kff_final['Expansion Implementation Date'],
    format='%m/%d/%Y',
    errors='coerce'
)

# Keep and rename relevant columns
kff_final = kff_final[['Location', 'expanded', 'date_adopted']].rename(
    columns={'Location': 'State'}
)

print('Expansion status counts:')
print(kff_final['expanded'].value_counts())
kff_final.head(10)

Expansion status counts:
expanded
True     41
False    10
Name: count, dtype: int64


,State,expanded,date_adopted
0,Alabama,False,NaT
1,Alaska,True,2015-09-01
2,Arizona,True,2014-01-01
3,Arkansas,True,2014-01-01
4,California,True,2014-01-01
5,Colorado,True,2014-01-01
6,Connecticut,True,2014-01-01
7,Delaware,True,2014-01-01
8,District of Columbia,True,2014-01-01
9,Florida,False,NaT


In [6]:
# State Crosswalk
crosswalk = {
    'Alabama': 'AL', 'Alaska': 'AK', 'Arizona': 'AZ', 'Arkansas': 'AR',
    'California': 'CA', 'Colorado': 'CO', 'Connecticut': 'CT', 'Delaware': 'DE',
    'Florida': 'FL', 'Georgia': 'GA', 'Hawaii': 'HI', 'Idaho': 'ID',
    'Illinois': 'IL', 'Indiana': 'IN', 'Iowa': 'IA', 'Kansas': 'KS',
    'Kentucky': 'KY', 'Louisiana': 'LA', 'Maine': 'ME', 'Maryland': 'MD',
    'Massachusetts': 'MA', 'Michigan': 'MI', 'Minnesota': 'MN', 'Mississippi': 'MS',
    'Missouri': 'MO', 'Montana': 'MT', 'Nebraska': 'NE', 'Nevada': 'NV',
    'New Hampshire': 'NH', 'New Jersey': 'NJ', 'New Mexico': 'NM', 'New York': 'NY',
    'North Carolina': 'NC', 'North Dakota': 'ND', 'Ohio': 'OH', 'Oklahoma': 'OK',
    'Oregon': 'OR', 'Pennsylvania': 'PA', 'Rhode Island': 'RI', 'South Carolina': 'SC',
    'South Dakota': 'SD', 'Tennessee': 'TN', 'Texas': 'TX', 'Utah': 'UT',
    'Vermont': 'VT', 'Virginia': 'VA', 'Washington': 'WA', 'West Virginia': 'WV',
    'Wisconsin': 'WI', 'Wyoming': 'WY', 'District of Columbia': 'DC'
}

kff_final['state_abbr'] = kff_final['State'].map(crosswalk)

# Check for unmatched states
unmatched = kff_final[kff_final['state_abbr'].isna()]
if len(unmatched) > 0:
    print('WARNING - unmatched states:')
    print(unmatched[['State']])
else:
    print('All states matched successfully!')

kff_final.head()

All states matched successfully!


,State,expanded,date_adopted,state_abbr
0,Alabama,False,NaT,AL
1,Alaska,True,2015-09-01,AK
2,Arizona,True,2014-01-01,AZ
3,Arkansas,True,2014-01-01,AR
4,California,True,2014-01-01,CA


In [8]:
# Load HCRIS Data (built during homework 4)
hcris = pd.read_csv('../../../data/output/HCRIS_Data.txt', sep='\t', low_memory=False)
print('HCRIS raw shape:', hcris.shape)
print('data_source values:', hcris['data_source'].value_counts().to_dict())
print('year range:', hcris['year'].min(), '-', hcris['year'].max())

HCRIS raw shape: (63115, 44)
data_source values: {'v2010': 56138, 'v1996': 6121}
year range: 2008 - 2020


In [9]:
# Filter HCRIS Data to 2010-2018 and v2010 data source
hcris = hcris[
    (hcris['data_source'] == 'v2010') &
    (hcris['year'].between(2010, 2018))
].copy()

print('HCRIS filtered shape:', hcris.shape)
print('Years remaining:', sorted(hcris['year'].unique()))
print('States:', hcris['state'].nunique())

HCRIS filtered shape: (47634, 44)
Years remaining: [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018]
States: 55


In [10]:
# Merge HCRIS and KFF
hcris_merged = hcris.merge(
    kff_final[['state_abbr', 'expanded', 'date_adopted']],
    left_on='state',
    right_on='state_abbr',
    how='left'
).drop(columns='state_abbr')

print('Merged shape:', hcris_merged.shape)

missing = hcris_merged['expanded'].isna().sum()
print(f'Rows with missing expansion status: {missing:,} ({100*missing/len(hcris_merged):.1f}%)')

Merged shape: (47634, 46)
Rows with missing expansion status: 2,784 (5.8%)


In [12]:
# Creating the Analysis Variables
hcris_merged['expand_year'] = pd.to_datetime(hcris_merged['date_adopted']).dt.year
hcris_merged['expand_ever'] = hcris_merged['expanded'].fillna(False)
hcris_merged['uncomp_care_m'] = hcris_merged['uncomp_care'] / 1_000_000
hcris_merged['expand_status'] = (
    hcris_merged['expand_ever'] &
    (hcris_merged['year'] >= hcris_merged['expand_year'])
)

print('expand_ever distribution:')
print(hcris_merged['expand_ever'].value_counts())
print('\nexpand_year distribution (top 10):')
print(hcris_merged['expand_year'].value_counts().head(10))
print('\nSample of key variables:')
hcris_merged[['provider_number', 'state', 'year', 'expand_ever', 
              'expand_year', 'uncomp_care_m']].head(10)

expand_ever distribution:
expand_ever
True     31498
False    16136
Name: count, dtype: int64

expand_year distribution (top 10):
expand_year
2014.0    20132
2015.0     3047
2021.0     2187
2016.0     2061
2020.0     1518
2023.0     1448
2019.0     1105
Name: count, dtype: int64

Sample of key variables:


/var/folders/dr/bz36x93s5vs6lnjjq8z60zfm0000gp/T/ipykernel_48041/2819281459.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  hcris_merged['expand_ever'] = hcris_merged['expanded'].fillna(False)


,provider_number,state,year,expand_ever,expand_year,uncomp_care_m
0,10001,AL,2011,False,NaN,NaN
1,10001,AL,2012,False,NaN,NaN
2,10001,AL,2013,False,NaN,NaN
3,10001,AL,2014,False,NaN,NaN
4,10001,AL,2015,False,NaN,106.497012
5,10001,AL,2016,False,NaN,114.974571
6,10001,AL,2017,False,NaN,120.244764
7,10001,AL,2018,False,NaN,129.711672
8,10005,AL,2011,False,NaN,4.438056
9,10005,AL,2012,False,NaN,41.035167


In [13]:
# Saving the final dataset
hcris_final = hcris_merged[~hcris_merged['expanded'].isna()].copy()
print('Final dataset shape:', hcris_final.shape)

hcris_final.to_pickle('../../../data/output/hcris_merged.pkl')
print('Saved successfully!')

Final dataset shape: (44850, 50)
Saved successfully!


In [14]:
print('=== Final Dataset Summary ===')
print(f'Total hospital-year observations: {len(hcris_final):,}')
print(f'Unique hospitals: {hcris_final["provider_number"].nunique():,}')
print(f'Unique states: {hcris_final["state"].nunique()}')
print(f'Years covered: {sorted(hcris_final["year"].unique())}')
print(f'Ever-expanders: {hcris_final[hcris_final["expand_ever"]]["state"].nunique()} states')
print(f'Never-expanders: {hcris_final[~hcris_final["expand_ever"]]["state"].nunique()} states')
print(f'\nMean uncompensated care (millions): ${hcris_final["uncomp_care_m"].mean():.2f}M')

=== Final Dataset Summary ===
Total hospital-year observations: 44,850
Unique hospitals: 6,546
Unique states: 51
Years covered: [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018]
Ever-expanders: 41 states
Never-expanders: 10 states

Mean uncompensated care (millions): $38.68M
